# Candidate recovery metric: days alive and out of hospital

Days alive and out of hospital folds mortality, length of stay and readmission into one ordered
number, which is why it has become the preferred composite in surgical trials. It keeps most of
the coverage of the EHR metrics while recovering some of the spread that a binary readmission
outcome throws away.

The audit questions are whether the components are all available, how much of the distribution
piles up at the maximum, and whether deaths are frequent enough to matter to the construction.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("days_alive_out_of_hospital")
CDR, CDR_INDEX = cfg.CDR, cfg.CDR_INDEX
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = srp_common.index_visit_cte(cfg) + f""",
    first_death AS (
        SELECT person_id, MIN(death_date) AS death_date
        FROM `{CDR}`.death
        GROUP BY person_id
    ),
    hospital_days AS (
        SELECT
            iv.person_id,
            SUM(GREATEST(0, DATE_DIFF(
                LEAST(vo.visit_end_date, DATE_ADD(iv.index_date, INTERVAL 30 DAY)),
                GREATEST(vo.visit_start_date, iv.index_date), DAY) + 1)) AS inpatient_days_30,
            SUM(GREATEST(0, DATE_DIFF(
                LEAST(vo.visit_end_date, DATE_ADD(iv.index_date, INTERVAL 90 DAY)),
                GREATEST(vo.visit_start_date, iv.index_date), DAY) + 1)) AS inpatient_days_90
        FROM index_visit AS iv
        JOIN `{CDR}`.visit_occurrence AS vo
          ON vo.person_id = iv.person_id
         AND vo.visit_concept_id IN (9201, 262)
         AND vo.visit_start_date <= DATE_ADD(iv.index_date, INTERVAL 90 DAY)
         AND vo.visit_end_date   >= iv.index_date
        WHERE iv.rn = 1
        GROUP BY iv.person_id
    ),
    daoh AS (
        SELECT
            iv.person_id,
            d.death_date IS NOT NULL AS died,
            IF(d.death_date IS NOT NULL
                 AND d.death_date <= DATE_ADD(iv.index_date, INTERVAL 30 DAY),
               0, GREATEST(0, 30 - COALESCE(h.inpatient_days_30, 0))) AS daoh_30,
            IF(d.death_date IS NOT NULL
                 AND d.death_date <= DATE_ADD(iv.index_date, INTERVAL 90 DAY),
               0, GREATEST(0, 90 - COALESCE(h.inpatient_days_90, 0))) AS daoh_90
        FROM index_visit AS iv
        LEFT JOIN hospital_days AS h ON h.person_id = iv.person_id
        LEFT JOIN first_death AS d ON d.person_id = iv.person_id
        WHERE iv.rn = 1
    )"""

# Death inside the horizon scores zero rather than being dropped -- that is the
# convention the metric is built on, and it is what makes DAOH a composite rather
# than a length-of-stay variant with survivors only.

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# 2-coverage

### Are all three components available, and for how many patients?

DAOH needs an index stay, every overlapping inpatient stay in the window, and the death table;
a missing death table would silently turn deaths into perfect recoveries. The death count is
reported here so that failure mode is visible rather than assumed away.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(*) FROM index_visit WHERE rn = 1)     AS patients_with_index_visit,
        (SELECT COUNT(*) FROM daoh)                         AS patients_with_daoh,
        (SELECT COUNTIF(died) FROM daoh)                    AS patients_with_death_record,
        (SELECT COUNTIF(daoh_30 = 0) FROM daoh)             AS daoh30_zero,
        (SELECT COUNTIF(daoh_30 = 30) FROM daoh)            AS daoh30_maximum
"""

cov = cached_gbq(coverage_sql, "daoh_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])
patients_with_daoh = int(cov["patients_with_daoh"])

for label in ["patients_with_index_visit", "patients_with_daoh",
              "patients_with_death_record", "daoh30_zero", "daoh30_maximum"]:
    n = int(cov[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:28s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

print()
print("--> daoh30_maximum is the ceiling effect: patients who spent no day in hospital "
      "after the index stay ended and did not die.")

# 3-distribution

### How is DAOH-30 distributed, and does a ceiling swallow it?

A composite that is 30 for most of the cohort behaves like a rare-event binary outcome rather
than a continuous one, and should be modelled as such. The shape of the left tail is what carries
the recovery signal.

In [ ]:
dist_sql = CTES + """
    SELECT daoh_30 AS daoh_bucket, COUNT(*) AS n_patients
    FROM daoh
    GROUP BY daoh_bucket
    ORDER BY daoh_bucket
"""

daoh_dist = cached_gbq(dist_sql, "daoh_30_distribution")
daoh_dist["daoh_label"] = [str(int(v)) for v in daoh_dist["daoh_bucket"]]

srp_common.plot_distribution(
    cfg, daoh_dist, "daoh_label",
    title="Patients by days alive and out of hospital within 30 days of surgery",
    xlabel="DAOH-30 (days)", suppress=False, save_as="daoh_30")
srp_common.plot_distribution(
    cfg, daoh_dist, "daoh_label",
    title="Patients by days alive and out of hospital within 30 days of surgery",
    xlabel="DAOH-30 (days)", suppress=True, save_as="daoh_30")

ceiling_share = int(cov["daoh30_maximum"]) / patients_with_daoh
print(f"share sitting at the DAOH-30 ceiling: {ceiling_share:.1%}")

### Does the 90-day horizon recover any of the spread the 30-day one loses?

The longer horizon has more room below the ceiling, at the cost of more censoring and a weaker
link to the surgery itself. Comparing the two quantile sets is the cheapest way to see whether
that trade is worth making.

In [ ]:
quantile_sql = CTES + """
    SELECT
        COUNT(*)                                        AS n_patients,
        APPROX_QUANTILES(daoh_30, 100)[OFFSET(10)]      AS daoh30_p10,
        APPROX_QUANTILES(daoh_30, 100)[OFFSET(25)]      AS daoh30_p25,
        APPROX_QUANTILES(daoh_30, 100)[OFFSET(50)]      AS daoh30_p50,
        APPROX_QUANTILES(daoh_90, 100)[OFFSET(10)]      AS daoh90_p10,
        APPROX_QUANTILES(daoh_90, 100)[OFFSET(25)]      AS daoh90_p25,
        APPROX_QUANTILES(daoh_90, 100)[OFFSET(50)]      AS daoh90_p50,
        STDDEV(daoh_30)                                 AS daoh30_sd,
        STDDEV(daoh_90)                                 AS daoh90_sd
    FROM daoh
"""

daoh_stats = cached_gbq(quantile_sql, "daoh_quantiles").iloc[0]

for horizon in (30, 90):
    p10, p25, p50, sd = (daoh_stats[f"daoh{horizon}_{k}"]
                         for k in ("p10", "p25", "p50", "sd"))
    print(f"DAOH-{horizon:<3} p10={int(p10):>3}  p25={int(p25):>3}  median={int(p50):>3}  "
          f"sd={sd:6.2f}  sd as share of horizon={sd / horizon:6.2%}")

print()
print("--> The larger relative spread is the better-behaved target, all else equal.")

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="ehr",
    source_table="visit_occurrence + death",
    index_rule=INDEX_RULE,
    outcome_type="count",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_index_visit"]),
    patients_analysable=patients_with_daoh,
    coverage_share=patients_with_daoh / cohort_patients,
    horizon_days=30,
    median_value=float(daoh_stats["daoh30_p50"]),
    ceiling_share=float(ceiling_share),
    informative_missingness="no",
    leakage_risk="medium",
    notes=("Composite of mortality, length of stay and readmission, so it cannot be used "
           "alongside those as features. Death inside the horizon scores zero by convention."),
)